# 08_significance — one pre-specified significance test

Supervisor requirement (meeting 2026-07-29): a p-test showing that the
before/after change is significant. One-sided, alpha = 0.05.

**The unit of independence is the article, not the sentence.** Computing one
number per article and comparing groups is the whole of the fix; no special
machinery is needed.

Reads `sentences_tagged.jsonl` from 04_extract. Writes to
`analysis/significance/`. Touches no other notebook's output.

## Step 1: Setup, hypothesis and parameters

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

_cwd = Path().resolve()
ROOT = next((p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()), _cwd)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

INTERIM_DIR  = ROOT / 'data' / 'interim' / 'iran'
OUTPUT_DIR   = ROOT / 'data' / 'output' / 'iran'
DIR_SIG      = OUTPUT_DIR / 'analysis' / 'significance'
DIR_SIG.mkdir(parents=True, exist_ok=True)

TAGGED = INTERIM_DIR / 'sentences_tagged.jsonl'

# ---------------------------------------------------------------------------
# ONE pre-specified test, fixed before looking at the result, so there is no
# multiple-comparison correction to make and nothing to explain away.
#
#   H0: the per-article share of military_action links does not differ
#       between the buildup and the climax.
#   H1: it is HIGHER in the climax.               (one-sided, alpha = 0.05)
#
# The unit of independence is the ARTICLE, not the sentence. Sentences inside
# one article share an author and a frame; a test over the full sentence set
# would treat them as independent and return a meaninglessly small p-value.
# ---------------------------------------------------------------------------
TARGET_CONCEPT   = 'military_action'
BUILDUP_WINDOWS  = ['buildup_mar', 'buildup_apr', 'buildup_may', 'buildup_jun1']
CLIMAX_WINDOWS   = ['climax_w1', 'climax_w2']
ALPHA            = 0.05
N_PERM           = 10_000
N_BOOT           = 10_000
SEED             = 42     # draws for the permutation/bootstrap resampling only

print(f'Tagged sentences : {TAGGED}')
print(f'Significance dir : {DIR_SIG}')
assert TAGGED.exists(), 'ERROR: run 04_extract first'


## Step 2: Per-article statistic under two link-counting conventions

In [ ]:
# Per-article statistic, under two link-counting conventions.
#
#   A (primary)    links = actors x concepts per sentence, so a sentence with
#                  2 actors and 2 concepts contributes 4 links. This is what
#                  the edge builder does, so the test measures the same object
#                  the networks are built from.
#   B (robustness) each concept occurrence in a sentence that has at least one
#                  actor counts once, however many actors the sentence holds.
rows = []
with open(TAGGED) as fh:
    for line in fh:
        d = json.loads(line)
        rows.append((d['article_id'], d['window'], len(d['actors']), d['concepts']))

per_article = {}
for art, win, n_act, concepts in rows:
    rec = per_article.setdefault(art, {'window': win, 'a_tot': 0, 'a_mil': 0,
                                       'b_tot': 0, 'b_mil': 0})
    if n_act == 0 or not concepts:
        continue
    rec['a_tot'] += n_act * len(concepts)
    rec['a_mil'] += n_act * concepts.count(TARGET_CONCEPT)
    rec['b_tot'] += len(concepts)
    rec['b_mil'] += concepts.count(TARGET_CONCEPT)

art_df = pd.DataFrame([
    {'article_id': a, 'window': r['window'],
     'links_A': r['a_tot'], 'mil_A': r['a_mil'],
     'links_B': r['b_tot'], 'mil_B': r['b_mil']}
    for a, r in per_article.items()
])
art_df = art_df[art_df.links_A > 0].copy()
art_df['share_A'] = art_df.mil_A / art_df.links_A
art_df['share_B'] = art_df.mil_B / art_df.links_B
art_df['phase'] = np.where(art_df.window.isin(BUILDUP_WINDOWS), 'buildup',
                  np.where(art_df.window.isin(CLIMAX_WINDOWS), 'climax', 'other'))

test_df = art_df[art_df.phase != 'other'].copy()
art_df.to_csv(DIR_SIG / 'per_article_shares.csv', index=False)
print(f'Articles with at least one actor-concept link : {len(art_df)}')
print(f'  buildup  : {(test_df.phase == "buildup").sum()}')
print(f'  climax   : {(test_df.phase == "climax").sum()}')
print(f'  aftermath (held out of the test)            : {(art_df.phase == "other").sum()}')


## Step 3: Mann-Whitney, permutation test, effect size, bootstrap CI

In [ ]:
def cohens_d(x, y):
    """Standardised mean difference, y minus x, pooled SD."""
    nx_, ny_ = len(x), len(y)
    sp = np.sqrt(((nx_ - 1) * x.var(ddof=1) + (ny_ - 1) * y.var(ddof=1))
                 / (nx_ + ny_ - 2))
    return (y.mean() - x.mean()) / sp


def run_test(col, label):
    b = test_df.loc[test_df.phase == 'buildup', col].to_numpy()
    c = test_df.loc[test_df.phase == 'climax',  col].to_numpy()
    obs = c.mean() - b.mean()

    # Mann-Whitney U, one-sided: is the climax stochastically larger?
    u, p_mw = mannwhitneyu(c, b, alternative='greater')

    # Permutation test: pool the articles, reshuffle the phase labels, rebuild
    # the null distribution of the mean difference. Assumption-free, and it is
    # the p-value the supervisor asked for.
    rng = np.random.default_rng(SEED)
    pool = np.concatenate([b, c])
    n_c = len(c)
    null = np.empty(N_PERM)
    for i in range(N_PERM):
        perm = rng.permutation(pool)
        null[i] = perm[:n_c].mean() - perm[n_c:].mean()
    p_perm = (np.sum(null >= obs) + 1) / (N_PERM + 1)   # add-one; never zero

    # Bootstrap CI on the difference of means
    boot = np.empty(N_BOOT)
    for i in range(N_BOOT):
        boot[i] = (rng.choice(c, n_c, replace=True).mean()
                   - rng.choice(b, len(b), replace=True).mean())
    lo, hi = np.percentile(boot, [2.5, 97.5])

    return ({
        'convention': label, 'n_buildup': len(b), 'n_climax': len(c),
        'mean_buildup': b.mean(), 'mean_climax': c.mean(),
        'median_buildup': float(np.median(b)), 'median_climax': float(np.median(c)),
        'diff_means': obs, 'ci95_low': lo, 'ci95_high': hi,
        'cohens_d': cohens_d(b, c), 'mannwhitney_U': u, 'p_mannwhitney': p_mw,
        'p_permutation': p_perm, 'n_permutations': N_PERM, 'alpha': ALPHA,
        'significant': bool(p_perm < ALPHA),
    }, b, c, null, obs)


res_A, bA, cA, null_A, obs_A = run_test('share_A', 'A: actors x concepts')
res_B, bB, cB, null_B, obs_B = run_test('share_B', 'B: concept presence')

sig_df = pd.DataFrame([res_A, res_B])
sig_df.to_csv(DIR_SIG / 'significance_military_action.csv', index=False)
print('Saved: significance_military_action.csv\n')
print(sig_df.T.to_string())
print(f'\nPRIMARY RESULT (convention A), per-article share of {TARGET_CONCEPT}:')
print(f'  {res_A["mean_buildup"]:.3f} in the buildup, {res_A["mean_climax"]:.3f} '
      f'at the climax, a difference of {res_A["diff_means"]:+.3f}')
print(f'  95% bootstrap CI [{res_A["ci95_low"]:.3f}, {res_A["ci95_high"]:.3f}]   '
      f"Cohen's d = {res_A['cohens_d']:.2f}")
print(f'  one-sided permutation p = {res_A["p_permutation"]:.5f}   '
      f'Mann-Whitney p = {res_A["p_mannwhitney"]:.3g}')


## Step 4: Figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
TEAL, PURPLE = '#1f8fa6', '#7e57c2'

ax = axes[0]
bins = np.linspace(0, 1, 26)
ax.hist(bA, bins=bins, alpha=.62, color=TEAL,   density=True,
        label=f'buildup (n={len(bA)})')
ax.hist(cA, bins=bins, alpha=.62, color=PURPLE, density=True,
        label=f'climax (n={len(cA)})')
ax.axvline(bA.mean(), color=TEAL,   ls='--', lw=2)
ax.axvline(cA.mean(), color=PURPLE, ls='--', lw=2)
ax.set_xlabel(f'per-article share of {TARGET_CONCEPT} links')
ax.set_ylabel('density')
ax.set_title(f'(a) Article-level distributions\nmeans {bA.mean():.3f} vs '
             f'{cA.mean():.3f}', fontsize=10)
ax.legend(fontsize=8); ax.grid(axis='y', alpha=.3)

ax = axes[1]
ax.hist(null_A, bins=60, color='#bdbdbd', edgecolor='none')
ax.axvline(obs_A, color='#d62728', lw=2.4,
           label=f'observed {obs_A:+.3f}\nnever reached in {N_PERM:,} shuffles')
ax.set_xlabel('difference of means under the null (phase labels permuted)')
ax.set_ylabel('count')
ax.set_title(f'(b) Permutation null, {N_PERM:,} shuffles', fontsize=10)
ax.legend(fontsize=8, loc='upper left'); ax.grid(axis='y', alpha=.3)

fig.suptitle(f'Buildup vs climax: per-article share of {TARGET_CONCEPT} links',
             fontsize=12)
plt.tight_layout()
out = DIR_SIG / 'significance_military_action.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out.name}')


## Step 5: Validation checkpoint

In [ ]:
print('VALIDATION CHECKPOINT (08_significance):')
print(f'  Articles in test          : {len(test_df)}  '
      f'({(test_df.phase=="buildup").sum()} buildup / '
      f'{(test_df.phase=="climax").sum()} climax)')
print(f'  Unit of independence      : article')
print(f'  Pre-specified hypothesis  : one-sided, climax > buildup, alpha = {ALPHA}')
print(f'  Tests run                 : 1 (no multiplicity correction needed)')
print()
for r in (res_A, res_B):
    verdict = 'REJECT H0' if r['significant'] else 'fail to reject H0'
    print(f'  [{r["convention"]:22s}]  d = {r["cohens_d"]:.2f}   '
          f'diff = {r["diff_means"]:+.3f}   p_perm = {r["p_permutation"]:.5f}   '
          f'{verdict}')
print()
for f in ('per_article_shares.csv', 'significance_military_action.csv',
          'significance_military_action.png'):
    print(f'    [{"OK" if (DIR_SIG / f).exists() else "MISS"}]  '
          f'analysis/significance/{f}')
print()
print('Report the EFFECT SIZE first and the p-value second. The p-value answers')
print('"could this be noise", which was never in doubt; d answers "how large".')
print('The value of this test is instrument validation: a known ground-truth')
print('shift (a war starts) is detected by the pipeline, which is what licenses')
print('the claims elsewhere whose answer is not obvious in advance.')
